# 03. Exploratory charts

Four views of standardized Medicare spending per beneficiary (`spend_pc_std`):

1. Spending vs. county median income, metro vs. rural (2024)
2. Spending over time, metro vs. rural (2014-2024)
3. County map (2024)
4. Sidebar: Miller County, GA by type of service (2024)

The last cell exports all four for the website.

`spend_pc_std` is CMS's price-standardized spending: it removes geographic
differences in what Medicare pays (wage indexes, etc.), so differences reflect
how much care is used rather than local prices. It is **not** adjusted for
inflation or for how sick beneficiaries are.

The connection is opened `read_only=True` so this notebook can run while another one has the database open read-only too.

In [ ]:
import json

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import plot_style  # shared fonts and colors (plot_style.py in this folder)
from plot_style import COLORS, MUTED, INK_2, SEQ_BLUE, NO_DATA, subtitle

con = duckdb.connect("../medicare.duckdb", read_only=True)

## 1. Spending vs. income, metro vs. rural (2024)

One dot per county. Lines are simple OLS fits within each group (no controls).

In [ ]:
scatter = con.sql("""
SELECT county_name, benes, spend_pc_std, median_income, poverty_rate, metro_rural
FROM merged
WHERE year = 2024 AND metro_rural IS NOT NULL
""").df()

scatter.groupby("metro_rural")[["spend_pc_std", "median_income"]].median().round(0)

In [ ]:
fig_scatter = px.scatter(
    scatter, x="median_income", y="spend_pc_std",
    color="metro_rural", color_discrete_map=COLORS,
    category_orders={"metro_rural": ["Metro", "Rural"]},
    trendline="ols", opacity=0.5,
    render_mode="svg",  # with >1,000 points Plotly switches to WebGL, which some browsers can't draw
    hover_name="county_name",
    hover_data={"metro_rural": False, "benes": ":,", "median_income": ":$,.0f",
                "spend_pc_std": ":$,.0f", "poverty_rate": ":.1f"},
    labels={"median_income": "Median household income", "spend_pc_std": "Spending per beneficiary",
            "benes": "Beneficiaries", "poverty_rate": "Poverty rate (%)"},
    title=subtitle("Higher-income counties spend slightly less per beneficiary, especially rural ones",
                   "Standardized Medicare spending per Original Medicare beneficiary, 2024. "
                   "One dot per county; lines are OLS fits."),
)
fig_scatter.update_traces(selector=dict(mode="markers"),
                          marker=dict(size=8, line=dict(width=1, color=plot_style.SURFACE)))
fig_scatter.update_traces(selector=dict(mode="lines"), line=dict(width=2.5), opacity=1)
fig_scatter.update_xaxes(tickprefix="$", tickformat=",.0f")
fig_scatter.update_yaxes(tickprefix="$", tickformat=",.0f")

# Label the most extreme county so readers know what the top dot is.
top = scatter.loc[scatter["spend_pc_std"].idxmax()]
fig_scatter.add_annotation(x=top["median_income"], y=top["spend_pc_std"],
                           text=f"{top['county_name']}", showarrow=True, arrowcolor=MUTED,
                           ax=50, ay=10, font=dict(color=INK_2, size=12))
fig_scatter.update_layout(height=560, legend_title_text="")
fig_scatter.show()

## 2. Spending over time, metro vs. rural

Each line is the beneficiary-weighted average across counties, i.e. total
standardized spending divided by total beneficiaries. Weighting by `benes`
means big counties count more, which is the right answer to "what does the
average beneficiary cost?" The unweighted median is shown in the table for comparison.

In [ ]:
trend = con.sql("""
SELECT year, metro_rural,
       SUM(spend_pc_std * benes) / SUM(benes) AS spend_weighted,
       MEDIAN(spend_pc_std)                   AS spend_median,
       COUNT(*)                               AS counties
FROM merged
WHERE metro_rural IS NOT NULL
GROUP BY ALL
ORDER BY metro_rural, year
""").df()

trend.pivot(index="year", columns="metro_rural", values=["spend_weighted", "spend_median"]).round(0)

In [ ]:
# Growth from first to last year, computed rather than typed in.
wide = trend.pivot(index="year", columns="metro_rural", values="spend_weighted")
growth = (wide.loc[2024] / wide.loc[2014] - 1) * 100
print(growth.round(1))

fig_trend = go.Figure()
for group in ["Metro", "Rural"]:
    d = trend[trend["metro_rural"] == group]
    fig_trend.add_scatter(
        x=d["year"], y=d["spend_weighted"], name=group, mode="lines+markers",
        line=dict(color=COLORS[group], width=2.5),
        marker=dict(size=8, line=dict(width=2, color=plot_style.SURFACE)),
        hovertemplate=group + ": $%{y:,.0f}<extra></extra>",
    )
    last = d.iloc[-1]  # direct label at the end of each line
    fig_trend.add_annotation(x=last["year"], y=last["spend_weighted"], text=f"<b>{group}</b>",
                             xanchor="left", xshift=10, showarrow=False,
                             font=dict(color=INK_2))

covid = trend[(trend["year"] == 2020) & (trend["metro_rural"] == "Metro")].iloc[0]
fig_trend.add_annotation(x=2020, y=covid["spend_weighted"], text="2020: care use fell during COVID-19",
                         showarrow=True, arrowcolor=MUTED, ax=0, ay=50, font=dict(color=MUTED, size=12))

fig_trend.update_layout(
    title=subtitle(f"Metro and rural spending per beneficiary both rose about {growth.mean():.0f}% from 2014 to 2024",
                   "Standardized Medicare spending per beneficiary, weighted by beneficiaries. "
                   "Nominal dollars."),
    hovermode="x unified", showlegend=True, height=500, margin=dict(r=80),
)
fig_trend.update_yaxes(tickprefix="$", tickformat=",.0f", title="Spending per beneficiary")
fig_trend.update_xaxes(dtick=1, title=None, range=[2013.6, 2024.4])  # fixed so narrow screens do not add empty years
fig_trend.show()

## 3. County map (2024)

The county shapes come from the Census 2023 cartographic boundary file
(`data/counties_2023.geojson`, built by `scripts/make_county_geojson.py`), which
has Connecticut's planning regions. Counties with fewer than 500 Original
Medicare beneficiaries are gray.

The color scale is capped at the 5th and 95th percentiles; otherwise a few
extreme counties (like Miller County, GA) would wash everything else into one shade.

In [ ]:
with open("../data/counties_2023.geojson") as f:
    counties_geo = json.load(f)

map_df = con.sql("""
SELECT fips, county_name, benes, spend_pc_std, metro_rural
FROM merged
WHERE year = 2024
""").df()

lo, hi = map_df["spend_pc_std"].quantile([0.05, 0.95])
print(f"Color range: ${lo:,.0f} to ${hi:,.0f}")

In [ ]:
fig_map = px.choropleth(
    map_df, geojson=counties_geo, locations="fips", color="spend_pc_std",
    color_continuous_scale=SEQ_BLUE, range_color=(lo, hi), scope="usa",
    hover_name="county_name",
    hover_data={"fips": False, "spend_pc_std": ":$,.0f", "benes": ":,", "metro_rural": True},
    labels={"spend_pc_std": "Spending per beneficiary", "benes": "Beneficiaries",
            "metro_rural": "Type"},
    title=subtitle("Medicare spending per beneficiary by county, 2024",
                   "Standardized dollars. Gray: fewer than 500 Original Medicare beneficiaries."),
)
fig_map.update_traces(marker_line_width=0.2, marker_line_color="white")
fig_map.update_geos(showland=True, landcolor=NO_DATA, showlakes=True, lakecolor=plot_style.SURFACE,
                    showsubunits=True, subunitcolor="white", subunitwidth=1,
                    bgcolor=plot_style.SURFACE)
fig_map.update_layout(
    height=600, margin=dict(l=10, r=10, t=90, b=10),
    coloraxis_colorbar=dict(title="", tickprefix="$", tickformat=",.0f", thickness=12, len=0.6),
)
fig_map.show()

## 4. Sidebar: Miller County, GA

Miller County is the highest-spending county in the data in 2023 and 2024.
This compares its 2024 spending by type of service with the national average.
The query reads the `raw` table directly because `merged` only keeps totals.

`UNPIVOT` is the reverse of the RUCC pivot in notebook 02: it turns one wide
row per place into one row per place and service type (like Stata's `reshape long`).

In [ ]:
miller = con.sql("""
WITH wide AS (
    SELECT CASE WHEN BENE_GEO_LVL = 'National' THEN 'United States' ELSE 'Miller County, GA' END AS place,
           TRY_CAST(BENES_OM_CNT AS INTEGER)                  AS benes,
           TRY_CAST(TOT_MDCR_STDZD_PYMT_PC AS DOUBLE)         AS total,
           TRY_CAST(SNF_MDCR_STDZD_PYMT_PC AS DOUBLE)         AS "Skilled nursing facility",
           TRY_CAST(IP_MDCR_STDZD_PYMT_PC AS DOUBLE)          AS "Inpatient hospital",
           TRY_CAST(OP_MDCR_STDZD_PYMT_PC AS DOUBLE)          AS "Outpatient hospital",
           TRY_CAST(EM_MDCR_STDZD_PYMT_PC AS DOUBLE)          AS "Physician visits",
           TRY_CAST(FQHC_RHC_MDCR_STDZD_PYMT_PC AS DOUBLE)    AS "Health centers and rural clinics"
    FROM raw
    WHERE YEAR = 2024 AND BENE_AGE_LVL = 'All'
      AND (BENE_GEO_LVL = 'National' OR (BENE_GEO_LVL = 'County' AND BENE_GEO_DESC = 'GA-Miller'))
)
UNPIVOT wide ON COLUMNS(* EXCLUDE (place, benes, total)) INTO NAME category VALUE spend
""").df()

# Everything not listed above becomes "All other services".
totals = con.sql("""
SELECT CASE WHEN BENE_GEO_LVL = 'National' THEN 'United States' ELSE 'Miller County, GA' END AS place,
       TRY_CAST(TOT_MDCR_STDZD_PYMT_PC AS DOUBLE) AS total
FROM raw
WHERE YEAR = 2024 AND BENE_AGE_LVL = 'All'
  AND (BENE_GEO_LVL = 'National' OR (BENE_GEO_LVL = 'County' AND BENE_GEO_DESC = 'GA-Miller'))
""").df().set_index("place")["total"]
other = (totals - miller.groupby("place")["spend"].sum()).rename("spend").reset_index()
other["category"] = "All other services"
miller = pd.concat([miller, other], ignore_index=True)

table = miller.pivot(index="category", columns="place", values="spend")
table["ratio"] = table["Miller County, GA"] / table["United States"]
table.sort_values("Miller County, GA", ascending=False).round(1)

In [ ]:
order = ["Skilled nursing facility", "Inpatient hospital", "Outpatient hospital",
         "Physician visits", "Health centers and rural clinics", "All other services"]
snf_ratio = table.loc["Skilled nursing facility", "ratio"]

fig_miller = go.Figure()
for place, color in [("United States", "#b5b3ad"), ("Miller County, GA", COLORS["Rural"])]:
    d = miller[miller["place"] == place].set_index("category").loc[order]
    fig_miller.add_bar(
        y=d.index, x=d["spend"], name=place, orientation="h", marker_color=color,
        text=[f"${v:,.0f}" for v in d["spend"]], textposition="outside",
        textfont=dict(color=INK_2, size=12), cliponaxis=False,
        hovertemplate=place + "<br>%{y}: $%{x:,.0f}<extra></extra>",
    )
fig_miller.update_layout(
    title=subtitle(f"Miller County, GA spent {snf_ratio:.0f} times the national average on skilled nursing in 2024",
                   f"Standardized Medicare spending per beneficiary by type of service. "
                   # &#36; is an HTML dollar sign. Two plain "$" in one string make Plotly treat the text between them as a LaTeX formula.
                   f"Total: &#36;{totals['Miller County, GA']:,.0f} vs. &#36;{totals['United States']:,.0f} nationally."),
    barmode="group", bargap=0.3, bargroupgap=0.08, height=480,
    legend=dict(traceorder="reversed"), margin=dict(l=230, r=60),
)
fig_miller.update_yaxes(autorange="reversed", showline=False, showgrid=False, tickfont=dict(color=INK_2))
fig_miller.update_xaxes(tickprefix="$", tickformat=",.0f", showline=False)
fig_miller.show()

Miller County had 559 Original Medicare beneficiaries in 2024, close to the
500 cutoff, so a small number of long or expensive stays can move its
per-person average a lot. Its spending is above the national average in most
categories. Skilled nursing is the largest gap in dollars (about $7,800 per
beneficiary). Health centers and rural clinics have the largest ratio (about 17
times the national average), though on a much smaller base.

## 5. Export charts for the website

Each chart is saved as its own HTML file for `gstenberg.org/medicare`.
The website copies drop the Plotly title (the page has its own headings) and
fill whatever box they are placed in. `include_plotlyjs="cdn"` loads the
Plotly library from the web instead of copying it into every file.

In [ ]:
import os

SITE = "../site/medicare/charts"
os.makedirs(SITE, exist_ok=True)

# Lets the chart fill the full height of the iframe it sits in.
FILL = "<style>html,body{height:100%;margin:0;background:#fcfcfb}body>div{height:100%}</style>"

for name, fig in {"income": fig_scatter, "trend": fig_trend, "map": fig_map, "miller": fig_miller}.items():
    web = go.Figure(fig)  # a copy, so the notebook version keeps its title
    web.update_layout(title=None, height=None, autosize=True, margin_t=50)
    if name == "miller":  # wrap long labels so the bars keep their width on phones
        def wrap(label):  # break at the space closest to the middle
            if len(label) <= 12:
                return label
            spaces = [i for i, ch in enumerate(label) if ch == " "]
            i = min(spaces, key=lambda i: abs(i - len(label) / 2))
            return label[:i] + "<br>" + label[i + 1:]
        web.update_yaxes(tickvals=order, ticktext=[wrap(c) for c in order], automargin=True)
        web.update_layout(margin_l=10)
    path = f"{SITE}/{name}.html"
    web.write_html(path, include_plotlyjs="cdn", full_html=True, default_height="100%",
                   config={"displayModeBar": False, "responsive": True})
    with open(path) as f:
        html = f.read().replace("</head>", FILL + "</head>", 1)
    with open(path, "w") as f:
        f.write(html)
    print(f"{path}  ({os.path.getsize(path) / 1e6:.1f} MB)")

In [ ]:
con.close()